In [9]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import pickle
from sklearn.model_selection import train_test_split

In [7]:
with open("C:\\Users\\Neela\\Documents\\GitHub\\EntityAspectLinking\\Experiment_trainsmall\\picklefiles\\baselinedataset_trainsmall.pkl", 'rb') as f:
    dataset = pickle.load(f)

In [8]:
#Undersampling to meet class imbalances for class 0 and 1
X = dataset[:,:-1]
sc = StandardScaler()
X = sc.fit_transform(X)
y = dataset[:,-1]
undersample = RandomUnderSampler(sampling_strategy='majority')
X, y = undersample.fit_resample(X, y)

In [13]:
#Train test split
X_train,X_test,y_train,y_test = train_test_split(X, y,test_size = 0.3, random_state = 42)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'objective': 'binary:logistic',  
    'eval_metric': 'logloss',        
    'max_depth': 3,                  
    'learning_rate': 0.1,            
    'subsample': 0.8,                
    'colsample_bytree': 0.8,         
    'seed': 42                       
}
num_rounds = 100  
model = xgb.train(params, dtrain, num_rounds)
y_pred = model.predict(dtest)
y_pred_binary = [1 if pred > 0.5 else 0 for pred in y_pred]  # Convert probabilities to binary predictions

print("Accuracy:", accuracy_score(y_test, y_pred_binary))
print("\nClassification Report:\n", classification_report(y_test, y_pred_binary))

Accuracy: 0.7862988784480146

Classification Report:
               precision    recall  f1-score   support

         0.0       0.78      0.81      0.79      1666
         1.0       0.80      0.76      0.78      1633

    accuracy                           0.79      3299
   macro avg       0.79      0.79      0.79      3299
weighted avg       0.79      0.79      0.79      3299



In [12]:
#Implementing Light GBM
import lightgbm as lgb
from sklearn.metrics import  roc_auc_score

# Create a LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

params = {
    'objective': 'binary',  # for binary classification 
    'metric': 'auc', # area under the curve
    'boosting_type': 'gbdt',    # traditional Gradient Boosting Decision Tree
    'num_leaves': 31,           # number of leaves in one tree
    'learning_rate': 0.05,      # learning rate
    'feature_fraction': 0.9,    # feature fraction
    'bagging_fraction': 0.8,    # bagging fraction
    'bagging_freq': 5,          # bagging frequency
    'verbose': 0,               # 0 for silent mode
}
# Train the model
num_round = 100  # Number of boosting rounds
bst = lgb.train(params, train_data, num_round, valid_sets = [test_data])
# Make predictions on the test set
y_pred_prob_lgb = bst.predict(X_test, num_iteration=bst.best_iteration)
y_pred_lgb = [1 if pred > 0.5 else 0 for pred in y_pred_prob_lgb]  # Convert probabilities to binary predictions

#Model evaluation 
accuracy = accuracy_score(y_test, y_pred_lgb)
roc_auc = roc_auc_score(y_test, y_pred_prob_lgb)

print(f'Accuracy on Test Set: {accuracy:.4f}')
print(f'ROC AUC on Test Set: {roc_auc:.4f}')
print("\nClassification Report:\n", classification_report(y_test, y_pred_lgb))


[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.072686 seconds.
You can set `force_col_wise=true` to remove the overhead.
[1]	valid_0's auc: 0.767529
[2]	valid_0's auc: 0.804291
[3]	valid_0's auc: 0.828687
[4]	valid_0's auc: 0.834617
[5]	valid_0's auc: 0.839362
[6]	valid_0's auc: 0.848484
[7]	valid_0's auc: 0.853238
[8]	valid_0's auc: 0.854712
[9]	valid_0's auc: 0.857599
[10]	valid_0's auc: 0.861996
[11]	valid_0's auc: 0.864628
[12]	valid_0's auc: 0.866452
[13]	valid_0's auc: 0.868
[14]	valid_0's auc: 0.873394
[15]	valid_0's auc: 0.875059
[16]	valid_0's auc: 0.87713
[17]	valid_0's auc: 0.87933
[18]	valid_0's auc: 0.879885
[19]	valid_0's auc: 0.881453
[20]	valid_0's auc: 0.882013
[21]	valid_0's auc: 0.883602
[22]	valid_0's auc: 0.884058
[23]	valid_0's auc: 0.8855
[24]	valid_0's auc: 0.886701
[25]	valid_0's auc: 0.887866
[26]	valid_0's auc: 0.888804
[27]	valid_0's auc: 0.888983
[28]	valid_0's auc: 0.890258
[29]	valid_0's auc: 0.890878
[30]	valid